In [ ]:
# In[1]:

import pandas as pd
import pytz

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrame variables (kept in kernel for later steps)
metric_df = pd.read_csv('metric.csv', low_memory=False)
trace_df = pd.read_csv('trace.csv', low_memory=False)
log_df = pd.read_csv('log.csv', low_memory=False)
error_logs_df = pd.read_csv('error_logs.csv', low_memory=False)

# Helper to add timezone-aware datetime column (reuse variable names)
def add_dt_column(df, ts_col='timestamp'):
    # Convert epoch seconds to timezone-aware datetimes in Asia/Shanghai (UTC+8)
    # Localize as UTC then convert to Asia/Shanghai
    dt = pd.to_datetime(df[ts_col], unit='s', errors='coerce').dt.tz_localize('UTC').dt.tz_convert(tz)
    df['_ts_dt'] = dt
    return df

metric_df = add_dt_column(metric_df)
trace_df = add_dt_column(trace_df)
log_df = add_dt_column(log_df)
error_logs_df = add_dt_column(error_logs_df)

# Helper to build compact summary dict for tabular telemetry
def build_summary(df, id_col='cmdb_id', name_col=None, sample_rows=None):
    total_rows = int(df.shape[0])
    unique_ids = df[id_col].dropna().unique()
    unique_ids_list = list(map(str, unique_ids[:50]))
    unique_id_count = int(len(unique_ids))
    name_list = None
    if name_col and name_col in df.columns:
        unique_names = df[name_col].dropna().unique()
        name_list = list(map(str, unique_names[:50]))
    min_ts = df['_ts_dt'].min()
    max_ts = df['_ts_dt'].max()
    res = {
        'total_rows': total_rows,
        'unique_cmdb_id_count': unique_id_count,
        'unique_cmdb_id_sample_up_to_50': unique_ids_list,
        f'unique_{name_col}_sample_up_to_50' if name_col else 'name_field': name_list,
        'min_timestamp_Asia/Shanghai': pd.Timestamp(min_ts) if pd.notnull(min_ts) else None,
        'max_timestamp_Asia/Shanghai': pd.Timestamp(max_ts) if pd.notnull(max_ts) else None
    }
    if sample_rows:
        res['sample_rows'] = sample_rows
    return res

# Build summaries
metric_summary = build_summary(metric_df, id_col='cmdb_id', name_col='kpi_name')
trace_summary = build_summary(trace_df, id_col='cmdb_id', name_col='trace_name')
log_summary = build_summary(log_df, id_col='cmdb_id', name_col='log_name')

# For error_logs, include up to 5 sample rows (timestamp, cmdb_id, message) and small summary
error_sample = error_logs_df.loc[:, ['_ts_dt', 'cmdb_id', 'message']].head(5).rename(columns={'_ts_dt':'timestamp_Asia/Shanghai'})
error_logs_summary = build_summary(error_logs_df, id_col='cmdb_id')
error_logs_summary['sample_rows_up_to_5'] = error_sample

# Display compact summaries (variables returned for IPython)
metric_summary, trace_summary, log_summary, error_logs_summary

```
Out[1]:
```
```python
# Compact plain-English summary string (displayed for the user)
summary = (
    "Summary of loaded telemetry files (timestamps shown in Asia/Shanghai UTC+8):\n\n"
    "metric.csv:\n"
    "- Total rows: 147,972\n"
    "- Unique cmdb_id count: 59\n"
    "- Sample of unique cmdb_id (up to 50): adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice, emailservice-0, emailservice-1, emailservice-2, emailservice2-0, frontend, frontend-0, frontend-1, frontend-2, frontend2-0, node-1, node-2, node-3, node-4, node-5, node-6, paymentservice, paymentservice-0, paymentservice-1, paymentservice-2, paymentservice2-0, productcatalogservice, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, productcatalogservice2-0, recommendationservice, recommendationservice-0, recommendationservice-1\n"
    "- Sample of unique kpi_name (up to 50 shown)\n"
    "- Time range: 2022-03-21 10:00:00 to 2022-03-21 10:29:00\n\n"
    "trace.csv:\n"
    "- Total rows: 35,896\n"
    "- Unique cmdb_id count: 40\n"
    "- Sample of unique cmdb_id (up to 50): adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice-0, emailservice-1, emailservice-2, emailservice2-0, frontend-0, frontend-1, frontend-2, frontend2-0, paymentservice-0, paymentservice-1, paymentservice-2, paymentservice2-0, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, productcatalogservice2-0, recommendationservice-0, recommendationservice-1, recommendationservice-2, recommendationservice2-0, shippingservice-0, shippingservice-1, shippingservice-2, shippingservice2-0\n"
    "- Sample of trace_name types (up to 50 shown), e.g., trace.self.duration_mean, trace.from_frontend-0.duration_mean, trace.from_checkoutservice-0.duration_p95, trace.from_root.row_count, etc.\n"
    "- Time range: 2022-03-21 10:00:00 to 2022-03-21 10:29:00\n\n"
    "log.csv:\n"
    "- Total rows: 1,814\n"
    "- Unique cmdb_id count: 31\n"
    "- Sample of unique cmdb_id (up to 50): adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, cartservice-2, checkoutservice-0, checkoutservice-1, checkoutservice-2, currencyservice-0, currencyservice-1, currencyservice-2, emailservice-0, emailservice-1, emailservice-2, frontend-0, frontend-1, frontend-2, paymentservice-0, paymentservice-1, paymentservice-2, productcatalogservice-0, productcatalogservice-1, productcatalogservice-2, recommendationservice-0, recommendationservice-1, recommendationservice-2, redis-cart-0, shippingservice-0, shippingservice-1, shippingservice-2\n"
    "- Unique log_name examples: log.error_count, log.row_count\n"
    "- Time range: 2022-03-21 10:00:00 to 2022-03-21 10:29:00\n\n"
    "error_logs.csv:\n"
    "- Total rows: 1,324\n"
    "- Unique cmdb_id count: 3 (frontend-0, frontend-2, frontend-1)\n"
    "- Up to 5 sample rows (timestamp, cmdb_id, message) show repeated warnings such as 'severity: warning, message: failed to retrieve...'\n"
    "- Time range: 2022-03-21 10:00:00 to 2022-03-21 10:29:59\n\n"
    "Full DataFrames are retained in kernel variables metric_df, trace_df, log_df, and error_logs_df for further analysis."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'total_rows': 147972, 'unique_cmdb_id_count': 59, 'unique_cmdb_id_sample_up_to_50': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'unique_kpi_name_sample_up_to_50': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'min_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:00:00+0800', tz='Asia/Shanghai'), 'max_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:29:00+0800', tz='Asia/Shanghai')}, {'total_rows': 35896, 'unique_cmdb_id_count': 40, 'unique_cmdb_id_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'], 'unique_trace_name_sample_up_to_50': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95'], 'min_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:00:00+0800', tz='Asia/Shanghai'), 'max_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:29:00+0800', tz='Asia/Shanghai')}, {'total_rows': 1814, 'unique_cmdb_id_count': 31, 'unique_cmdb_id_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'redis-cart-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2'], 'unique_log_name_sample_up_to_50': ['log.error_count', 'log.row_count'], 'min_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:00:00+0800', tz='Asia/Shanghai'), 'max_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:29:00+0800', tz='Asia/Shanghai')}, {'total_rows': 1324, 'unique_cmdb_id_count': 3, 'unique_cmdb_id_sample_up_to_50': ['frontend-0', 'frontend-2', 'frontend-1'], 'name_field': None, 'min_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:00:00+0800', tz='Asia/Shanghai'), 'max_timestamp_Asia/Shanghai': Timestamp('2022-03-21 10:29:59+0800', tz='Asia/Shanghai'), 'sample_rows_up_to_5':     timestamp_Asia/Shanghai     cmdb_id                                            message
0 2022-03-21 10:00:00+08:00  frontend-0  severity: warning, message: failed to retrieve...
1 2022-03-21 10:00:02+08:00  frontend-2  severity: warning, message: failed to retrieve...
2 2022-03-21 10:00:03+08:00  frontend-0  severity: warning, message: failed to retrieve...
3 2022-03-21 10:00:06+08:00  frontend-1  severity: warning, message: failed to retrieve...
4 2022-03-21 10:00:07+08:00  frontend-2  severity: warning, message: failed to retrieve...})```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Define incident window (inclusive) in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Group by series (cmdb_id, kpi_name) and compute global percentiles and counts (use entire series)
percentiles = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    p95=lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().shape[0] > 0 else np.nan,
    p90=lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().shape[0] > 0 else np.nan,
    p50=lambda x: np.nanpercentile(x.dropna(), 50) if x.dropna().shape[0] > 0 else np.nan,
    p5=lambda x: np.nanpercentile(x.dropna(), 5) if x.dropna().shape[0] > 0 else np.nan,
    total_count='count'
)

# Get min and max timestamps per series (from the full data)
ts_range = metric_df.groupby(['cmdb_id', 'kpi_name'])['_ts_dt'].agg(
    min_timestamp='min',
    max_timestamp='max'
)

# Combine percentiles and timestamp ranges
thresholds_df = percentiles.merge(ts_range, left_index=True, right_index=True).reset_index()

# Compute in-window counts (count of points within the incident window inclusive)
in_window_mask = (metric_df['_ts_dt'] >= window_start) & (metric_df['_ts_dt'] <= window_end)
in_window_counts = metric_df.loc[in_window_mask].groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('in_window_count')
# Merge, fill missing with 0
thresholds_df = thresholds_df.merge(in_window_counts.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')
thresholds_df['in_window_count'] = thresholds_df['in_window_count'].fillna(0).astype(int)

# Ensure numeric columns have proper dtypes
for col in ['p95','p90','p50','p5']:
    thresholds_df[col] = pd.to_numeric(thresholds_df[col], errors='coerce')

thresholds_df['total_count'] = thresholds_df['total_count'].astype(int)

# Keep full result in kernel variable thresholds_df (as required)
# Prepare compact top-20 view sorted by p95 descending
thresholds_top20 = thresholds_df.sort_values('p95', ascending=False).head(20)[
    ['cmdb_id','kpi_name','p95','p90','p50','p5','total_count','in_window_count','min_timestamp','max_timestamp']
]

thresholds_df, thresholds_top20

```
Out[2]:
```
I computed global percentiles (P95, P90, P50, P5), total sample counts, min/max timestamps, and the count of samples inside the incident window (2022-03-21 10:00:00 — 10:30:00 Asia/Shanghai) for every (cmdb_id, kpi_name) series and stored the full result as thresholds_df in the kernel.

Key points from the results (top 20 series by P95):

- Number of series processed: 4,942 (each row in thresholds_df is a unique (cmdb_id, kpi_name)).
- Most series have total_count = 30 and in_window_count = 30, i.e., they have 30 samples and all samples fall inside the incident window (2022-03-21 10:00 to 10:29 Asia/Shanghai).
- The top P95 values are dominated by runtime/JVM and node filesystem/disk metrics, for example:
  - runtime.java_lang_Runtime_StartTime for adservice2 and adservice: P95 ≈ 1.6478e+12 (these are epoch-like start-time values).
  - runtime.java_lang_OperatingSystem_ProcessCpuTime and threading CPU-time metrics show very large P95 values (e.g., ~9.7e+11).
  - Node metrics such as system.disk.total and system.disk.used appear with P95 in the billions (e.g., ~7.6e+09).
  - Some mesh/jaeger-related metrics show large variability (examples in the full table).
- Time coverage for these top entries: min_timestamp = 2022-03-21 10:00:00+08:00 and max_timestamp = 2022-03-21 10:29:00+08:00 (consistent across the top rows).

If you want, I can:
- Show the stored thresholds_df shape and a compact preview (e.g., thresholds_df.head(10) or the full top20 table again),
- Filter thresholds_df for specific services or KPI types (CPU, memory, network),
- Compute anomaly thresholds relative to historical variability for selected KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name           p95           p90           p50            p5  total_count             min_timestamp             max_timestamp  in_window_count
0              adservice                                     app.grpc.count    203.550000    198.500000    168.500000    149.000000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
1              adservice                                       app.grpc.mrt      0.002666      0.002629      0.002506      0.002449           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
2              adservice                                        app.grpc.rr    100.000000    100.000000    100.000000    100.000000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
3              adservice                                        app.grpc.sr     77.137164     76.737460     74.931694     71.948413           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
4              adservice                                     app.http.count     24.000000     24.000000     24.000000     24.000000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
...                  ...                                                ...           ...           ...           ...           ...          ...                       ...                       ...              ...
4937  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     19.825000     19.000000     14.250000      6.225000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
4938  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  56688.750000  56315.000000  41587.500000  16723.750000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
4939  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001364      0.000525      0.000000      0.000000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
4940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     18.500000     18.050000     13.750000      5.725000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
4941  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1818.362500   1791.000000   1343.250000    569.637500           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30

[4942 rows x 10 columns],          cmdb_id                                           kpi_name           p95           p90           p50            p5  total_count  in_window_count             min_timestamp             max_timestamp
841   adservice2                runtime.java_lang_Runtime_StartTime  1.647822e+12  1.647822e+12  1.647822e+12  1.647822e+12           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
250    adservice                runtime.java_lang_Runtime_StartTime  1.647523e+12  1.647523e+12  1.647523e+12  1.647523e+12           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  9.706462e+11  9.703853e+11  9.683038e+11  9.660180e+11           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  9.427890e+10  9.424508e+10  9.404019e+10  9.379812e+10           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  9.170879e+10  9.168267e+10  9.147125e+10  9.123579e+10           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  7.475849e+10  7.473661e+10  7.456760e+10  7.437653e+10           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
835   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  4.538883e+10  4.497858e+10  4.076458e+10  3.741962e+10           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3367      node-4                                  system.disk.total  7.616300e+09  7.616300e+09  7.616300e+09  7.616300e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3371      node-4                             system.fs.inodes.total  6.550459e+09  6.548647e+09  6.545585e+09  6.542453e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3369      node-4                              system.fs.inodes.free  6.547868e+09  6.546056e+09  6.542995e+09  6.539862e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3250      node-2                                  system.disk.total  5.867011e+09  5.867011e+09  5.867011e+09  5.867011e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3308      node-3                                  system.disk.total  5.867011e+09  5.867011e+09  5.867011e+09  5.867011e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3191      node-1                                  system.disk.total  5.867011e+09  5.867011e+09  5.867011e+09  5.867011e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3368      node-4                                   system.disk.used  5.784316e+09  5.784201e+09  5.783533e+09  5.782314e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3195      node-1                             system.fs.inodes.total  4.922627e+09  4.921211e+09  4.918961e+09  4.916594e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3254      node-2                             system.fs.inodes.total  4.922626e+09  4.921217e+09  4.918962e+09  4.916594e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3312      node-3                             system.fs.inodes.total  4.922494e+09  4.920912e+09  4.919113e+09  4.916508e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3193      node-1                              system.fs.inodes.free  4.920684e+09  4.919269e+09  4.917019e+09  4.914651e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3252      node-2                              system.fs.inodes.free  4.920654e+09  4.919245e+09  4.916990e+09  4.914621e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3310      node-3                              system.fs.inodes.free  4.920544e+09  4.918962e+09  4.917162e+09  4.914558e+09           30               30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing kernel variables: metric_df, thresholds_df
tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Filter metric_df to the incident window (inclusive)
window_mask = (metric_df['_ts_dt'] >= window_start) & (metric_df['_ts_dt'] <= window_end)
metric_window = metric_df.loc[window_mask].copy()

# Prepare series list with global thresholds from thresholds_df
series_thresholds = thresholds_df[['cmdb_id', 'kpi_name', 'p95', 'p5']].drop_duplicates().reset_index(drop=True)

# Merge thresholds into metric_window so each row carries its series thresholds
metric_win_with_thresh = metric_window.merge(series_thresholds, on=['cmdb_id', 'kpi_name'], how='right', suffixes=('', '_thr'))

# Helper to compute longest consecutive run of True values in a boolean array
def longest_run_bool(arr):
    if len(arr) == 0:
        return 0
    max_run = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

# We'll compute per-series aggregates using groups present in the merged DataFrame.
# Note: metric_win_with_thresh has NaNs for rows where there was no metric data in the window.
# We will iterate over series_thresholds to ensure every series in thresholds_df is included.
records = []
# For faster lookup, create an index of metric_window grouped by (cmdb_id,kpi_name)
grouped = metric_window.groupby(['cmdb_id', 'kpi_name'])

for _, row in series_thresholds.iterrows():
    cid = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['p95']
    p5 = row['p5']
    key = (cid, kpi)
    if key in grouped.groups:
        grp = grouped.get_group(key).sort_values('_ts_dt')
        vals = grp['value'].to_numpy()
        times = grp['_ts_dt'].to_numpy()
        total_count = int(len(vals))
        above_mask = vals >= p95
        below_mask = vals <= p5
        above_count = int(np.count_nonzero(above_mask))
        below_count = int(np.count_nonzero(below_mask))
        earliest_above_ts = pd.to_datetime(times[above_mask].min()) if above_count > 0 else pd.NaT
        earliest_below_ts = pd.to_datetime(times[below_mask].min()) if below_count > 0 else pd.NaT
        longest_run_above = longest_run_bool(above_mask)
        longest_run_below = longest_run_bool(below_mask)
    else:
        # no data in window for this series
        total_count = 0
        above_count = 0
        below_count = 0
        earliest_above_ts = pd.NaT
        earliest_below_ts = pd.NaT
        longest_run_above = 0
        longest_run_below = 0

    records.append({
        'cmdb_id': cid,
        'kpi_name': kpi,
        'p95': p95,
        'p5': p5,
        'total_count': total_count,
        'above_count': above_count,
        'below_count': below_count,
        'earliest_above_ts': earliest_above_ts,
        'earliest_below_ts': earliest_below_ts,
        'longest_run_above': longest_run_above,
        'longest_run_below': longest_run_below
    })

# Build anomalies_df and store in kernel
anomalies_df = pd.DataFrame.from_records(records)

# For sorting by severity, compute max_run column
anomalies_df['max_run'] = anomalies_df[['longest_run_above','longest_run_below']].max(axis=1)

# Prepare compact preview: top 20 rows sorted by max_run desc, then by above_count desc as tie-breaker
anomalies_top20 = anomalies_df.sort_values(['max_run','above_count'], ascending=[False, False]).head(20)[
    ['cmdb_id','kpi_name','p95','p5','total_count','above_count','below_count','earliest_above_ts','earliest_below_ts','longest_run_above','longest_run_below']
]

anomalies_df, anomalies_top20

```
Out[3]:
```
```python
# Plain-English compact summary (uses existing anomalies_df in kernel)
n_series = int(anomalies_df.shape[0])
n_with_data = int((anomalies_df['total_count'] > 0).sum())
n_full_window = int((anomalies_df['total_count'] == 30).sum())
n_with_above = int((anomalies_df['above_count'] > 0).sum())
n_with_below = int((anomalies_df['below_count'] > 0).sum())
max_run = int(anomalies_df['max_run'].max())

top_examples_df = anomalies_df.sort_values('max_run', ascending=False).head(5)[
    ['cmdb_id','kpi_name','max_run','above_count','below_count']
]
top_examples_lines = []
for _, r in top_examples_df.iterrows():
    top_examples_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | max_run={int(r['max_run'])}min, above={int(r['above_count'])}, below={int(r['below_count'])}")
top_examples_text = "; ".join(top_examples_lines)

summary = (
    f"Anomaly window summary (2022-03-21 10:00–10:30 Asia/Shanghai):\n"
    f"- Series processed: {n_series}\n"
    f"- Series with any data in window: {n_with_data}\n"
    f"- Series with full 30-minute coverage: {n_full_window}\n"
    f"- Series with at least one value >= global P95: {n_with_above}\n"
    f"- Series with at least one value <= global P5: {n_with_below}\n"
    f"- Longest observed consecutive run (above or below threshold): {max_run} minutes\n\n"
    f"Notable examples (top 5 by longest run): {top_examples_text}\n\n"
    f"Observations: many series (including several runtime/JVM and node disk metrics) have full coverage (30 samples) and some series "
    f"(e.g., adservice app.grpc.rr and app.http.count) show every point equal/at-threshold producing longest-run = 30. "
    f"Full per-series results are stored as anomalies_df in the kernel; thresholds_df and metric_df remain available for deeper inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name           p95            p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below  max_run
0              adservice                                     app.grpc.count    203.550000    149.000000           30            2            3 2022-03-21 10:24:00+08:00 2022-03-21 10:00:00+08:00                  1                  2        2
1              adservice                                       app.grpc.mrt      0.002666      0.002449           30            2            2 2022-03-21 10:10:00+08:00 2022-03-21 10:11:00+08:00                  1                  1        1
2              adservice                                        app.grpc.rr    100.000000    100.000000           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30
3              adservice                                        app.grpc.sr     77.137164     71.948413           30            2            2 2022-03-21 10:00:00+08:00 2022-03-21 10:07:00+08:00                  2                  1        2
4              adservice                                     app.http.count     24.000000     24.000000           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30
...                  ...                                                ...           ...           ...          ...          ...          ...                       ...                       ...                ...                ...      ...
4937  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     19.825000      6.225000           30            2            2 2022-03-21 10:07:00+08:00 2022-03-21 10:06:00+08:00                  1                  1        1
4938  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  56688.750000  16723.750000           30            2            2 2022-03-21 10:25:00+08:00 2022-03-21 10:00:00+08:00                  1                  2        2
4939  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001364      0.000000           30            2           22 2022-03-21 10:15:00+08:00 2022-03-21 10:00:00+08:00                  1                 11       11
4940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     18.500000      5.725000           30            3            2 2022-03-21 10:23:00+08:00 2022-03-21 10:00:00+08:00                  1                  2        2
4941  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1818.362500    569.637500           30            2            2 2022-03-21 10:23:00+08:00 2022-03-21 10:00:00+08:00                  1                  2        2

[4942 rows x 12 columns],       cmdb_id                                           kpi_name         p95          p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below
2   adservice                                        app.grpc.rr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
4   adservice                                     app.http.count        24.0        24.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
6   adservice                                        app.http.rr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
7   adservice                                        app.http.sr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount      4990.0      4990.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...      4990.0      4990.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount         0.0         0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
11  adservice             runtime.java_lang_ClassLoading_Verbose         0.0         0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...         1.0         1.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
13  adservice  runtime.java_lang_Compilation_TotalCompilation...     43858.0     43858.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...         3.0         3.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...       177.0       177.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         1.0         1.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         1.0         1.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...        63.0        63.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     56457.0     56457.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         3.0         3.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  22937600.0  22937600.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   7340032.0   7340032.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   3801088.0   3801088.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing kernel variables: anomalies_df, metric_df, thresholds_df
tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window (UTC+8)
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Ensure metric_window (data within incident window) is available / recreated
window_mask = (metric_df['_ts_dt'] >= window_start) & (metric_df['_ts_dt'] <= window_end)
metric_window = metric_df.loc[window_mask].copy()

# Filter anomalies_df to candidate faulty series
candidates_df = anomalies_df[
    (anomalies_df['max_run'] >= 3) &
    (anomalies_df['total_count'] > 0) &
    ((anomalies_df['above_count'] > 0) | (anomalies_df['below_count'] > 0))
].copy()

# Compute in-window min/max values per series
value_stats = metric_window.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    value_max_in_window='max',
    value_min_in_window='min'
).reset_index()

# Merge value stats into candidates_df
candidates_df = candidates_df.merge(value_stats, on=['cmdb_id', 'kpi_name'], how='left')

# Sort by severity (max_run desc, then above_count desc) and limit to top 50 for preview
candidates_preview = candidates_df.sort_values(['max_run','above_count'], ascending=[False, False]).head(50)[
    ['cmdb_id','kpi_name','p95','p5','total_count','above_count','below_count',
     'earliest_above_ts','earliest_below_ts','longest_run_above','longest_run_below','max_run',
     'value_max_in_window','value_min_in_window']
]

# Return stored full candidates_df and compact preview for display
candidates_df, candidates_preview

```
Out[4]:
```
```python
# Compact plain-English summary using existing kernel variables candidates_df and candidates_preview
n_candidates = int(candidates_df.shape[0])
n_services = int(candidates_df['cmdb_id'].nunique())
max_run = int(candidates_df['max_run'].max())

# Top 5 examples by max_run (tie-breaker above_count)
top5 = candidates_df.sort_values(['max_run','above_count'], ascending=[False, False]).head(5)[
    ['cmdb_id','kpi_name','max_run','above_count','below_count','value_max_in_window','value_min_in_window']
]

top5_lines = []
for _, r in top5.iterrows():
    top5_lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | max_run={int(r['max_run'])}min | above={int(r['above_count'])} | below={int(r['below_count'])} | max={r['value_max_in_window']} | min={r['value_min_in_window']}"
    )

summary = (
    f"Candidate faulty series summary (criteria: max_run >= 3, total_count>0, above_count>0 or below_count>0):\n"
    f"- Total candidate series: {n_candidates}\n"
    f"- Affected distinct cmdb_id: {n_services}\n"
    f"- Longest consecutive run observed among candidates: {max_run} minutes\n\n"
    f"Top examples (up to 5) by longest run:\n"
    + "\n".join(top5_lines) +
    "\n\n"
    "Notes:\n"
    "- Each candidate row represents one (cmdb_id, kpi_name) series with at least one breach and a consecutive run >= 3 minutes within 2022-03-21 10:00–10:30 Asia/Shanghai.\n"
    "- Full candidate results are stored in the kernel variable candidates_df for further inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name           p95       p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below  max_run  value_max_in_window  value_min_in_window
0              adservice                                        app.grpc.rr    100.000000    100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30           100.000000                100.0
1              adservice                                     app.http.count     24.000000     24.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30            24.000000                 24.0
2              adservice                                        app.http.rr    100.000000    100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30           100.000000                100.0
3              adservice                                        app.http.sr    100.000000    100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30           100.000000                100.0
4              adservice    runtime.java_lang_ClassLoading_LoadedClassCount   4990.000000   4990.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30          4990.000000               4990.0
...                  ...                                                ...           ...      ...          ...          ...          ...                       ...                       ...                ...                ...      ...                  ...                  ...
3291  shippingservice2-0             container.node-5.container_threads_max      0.000000      0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30             0.000000                  0.0
3292  shippingservice2-0  container.node-5.container_ulimits_soft.max_op...  65536.000000  65536.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30         65536.000000              65536.0
3293  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...      0.000525      0.0           30            3           27 2022-03-21 10:07:00+08:00 2022-03-21 10:00:00+08:00                  2                 13       13             0.002050                  0.0
3294  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      0.001575      0.0           30            3           12 2022-03-21 10:05:00+08:00 2022-03-21 10:00:00+08:00                  1                  3        3             0.004575                  0.0
3295  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001364      0.0           30            2           22 2022-03-21 10:15:00+08:00 2022-03-21 10:00:00+08:00                  1                 11       11             0.003050                  0.0

[3296 rows x 14 columns],       cmdb_id                                           kpi_name         p95          p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below  max_run  value_max_in_window  value_min_in_window
0   adservice                                        app.grpc.rr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                100.0                100.0
1   adservice                                     app.http.count        24.0        24.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                 24.0                 24.0
2   adservice                                        app.http.rr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                100.0                100.0
3   adservice                                        app.http.sr       100.0       100.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                100.0                100.0
4   adservice    runtime.java_lang_ClassLoading_LoadedClassCount      4990.0      4990.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30               4990.0               4990.0
..        ...                                                ...         ...         ...          ...          ...          ...                       ...                       ...                ...                ...      ...                  ...                  ...
45  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  41943040.0  41943040.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30           41943040.0           41943040.0
46  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  41943040.0  41943040.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30           41943040.0           41943040.0
47  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...        -1.0        -1.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                 -1.0                 -1.0
48  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...        -1.0        -1.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                 -1.0                 -1.0
49  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   5242880.0   5242880.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30            5242880.0            5242880.0

[50 rows x 14 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse candidates_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Aggregate by cmdb_id using only candidate rows in candidates_df
grp = candidates_df.groupby('cmdb_id')

def earliest_anomaly(series):
    # series is a DataFrameGroupBy for a single cmdb_id; we will examine earliest_above_ts and earliest_below_ts
    above_min = series['earliest_above_ts'].min() if 'earliest_above_ts' in series else pd.NaT
    below_min = series['earliest_below_ts'].min() if 'earliest_below_ts' in series else pd.NaT
    # Build list dropping NaT then take min, else NaT
    candidates = []
    if pd.notnull(above_min):
        candidates.append(above_min)
    if pd.notnull(below_min):
        candidates.append(below_min)
    return min(candidates) if candidates else pd.NaT

# Perform aggregations
candidate_components_df = grp.agg(
    candidate_kpi_count = ('kpi_name', 'nunique'),
    affected_total_points = ('total_count', 'sum'),
    total_above_count = ('above_count', 'sum'),
    total_below_count = ('below_count', 'sum'),
    max_run = ('max_run', 'max'),
    max_value_in_window = ('value_max_in_window', 'max'),
    min_value_in_window = ('value_min_in_window', 'min')
).reset_index()

# Compute earliest_anomaly_ts per cmdb_id
# We compute group-wise earliest across earliest_above_ts and earliest_below_ts columns
earliest_map = {}
for name, group in grp:
    above_min = group['earliest_above_ts'].min() if 'earliest_above_ts' in group else pd.NaT
    below_min = group['earliest_below_ts'].min() if 'earliest_below_ts' in group else pd.NaT
    times = []
    if pd.notnull(above_min):
        times.append(above_min)
    if pd.notnull(below_min):
        times.append(below_min)
    earliest_map[name] = min(times) if times else pd.NaT

# Map earliest_anomaly_ts into the aggregated dataframe
candidate_components_df['earliest_anomaly_ts'] = candidate_components_df['cmdb_id'].map(earliest_map)

# Reorder columns as requested
candidate_components_df = candidate_components_df[
    ['cmdb_id','candidate_kpi_count','max_run','earliest_anomaly_ts',
     'total_above_count','total_below_count','affected_total_points',
     'max_value_in_window','min_value_in_window']
]

# Prepare compact top-20 preview sorted by candidate_kpi_count desc then max_run desc
candidate_components_top20 = candidate_components_df.sort_values(
    ['candidate_kpi_count','max_run'], ascending=[False, False]
).head(20)

# Return full aggregated table and the compact preview (kernel will retain both)
candidate_components_df, candidate_components_top20

```
Out[5]:
```
```python
summary = (
    "Candidate components summary (based on candidate KPIs with consecutive runs >=3 minutes during 2022-03-21 10:00–10:30 Asia/Shanghai):\n\n"
    "- Number of affected components (cmdb_id) found: 59.\n"
    "- Most impacted components by distinct candidate KPIs:\n"
    "  * adservice: 305 KPIs (affected_total_points=9150, above_count=8883, below_count=9038, max_run=30, earliest_anomaly=2022-03-21 10:00+08:00)\n"
    "  * adservice2: 280 KPIs (affected_total_points=8400, above_count=7876, below_count=8004, max_run=30, earliest_anomaly=2022-03-21 10:00+08:00)\n"
    "  * frontend2-0: 107 KPIs; frontend-0: 98 KPIs; frontend-1: 96 KPIs; frontend-2: 82 KPIs (all with max_run=30 and earliest anomalies at 10:00+08:00)\n"
    "- Many components show max_run = 30 minutes (i.e., full-window consecutive breaches) indicating sustained deviations for those KPIs during the incident window.\n"
    "- Earliest anomaly timestamps for top components are predominantly 2022-03-21 10:00:00+08:00, meaning breaches were present at the start of the incident window.\n"
    "- Value ranges: per-component aggregated max/min in-window values vary widely (some KPIs are epoch-like or large JVM counters; some min values are negative or zero). See candidate_components_df for exact per-component min/max numbers.\n\n"
    "Notes and next steps:\n"
    "- The kernel contains the full candidate-level table (candidates_df) and the aggregated per-component table (candidate_components_df) for inspection.\n"
    "- Recommended next actions: inspect top components' KPI details (which specific KPIs are breached), check logs/traces for those components (e.g., frontend and adservice), and correlate with error_logs for root-cause analysis."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id  candidate_kpi_count  max_run       earliest_anomaly_ts  total_above_count  total_below_count  affected_total_points  max_value_in_window  min_value_in_window
0            adservice                  305       30 2022-03-21 10:00:00+08:00               8883               9038                   9150         1.647523e+12                 -1.0
1          adservice-0                   64       30 2022-03-21 10:00:00+08:00               1762               1864                   1920         1.647523e+09                  0.0
2          adservice-1                   65       30 2022-03-21 10:00:00+08:00               1782               1933                   1950         1.647523e+09                  0.0
3          adservice-2                   67       30 2022-03-21 10:00:00+08:00               1787               1877                   2010         1.647523e+09                  0.0
4           adservice2                  280       30 2022-03-21 10:00:00+08:00               7876               8004                   8400         1.647822e+12                 -1.0
..                 ...                  ...      ...                       ...                ...                ...                    ...                  ...                  ...
54     shippingservice                    2       30 2022-03-21 10:00:00+08:00                 60                 60                     60         1.000000e+02                100.0
55   shippingservice-0                   51       30 2022-03-21 10:00:00+08:00               1448               1508                   1530         1.647523e+09                  0.0
56   shippingservice-1                   52       30 2022-03-21 10:00:00+08:00               1479               1534                   1560         1.647523e+09                  0.0
57   shippingservice-2                   52       30 2022-03-21 10:00:00+08:00               1467               1514                   1560         1.647523e+09                  0.0
58  shippingservice2-0                   52       30 2022-03-21 10:00:00+08:00               1330               1466                   1560         1.647523e+09                  0.0

[59 rows x 9 columns],                      cmdb_id  candidate_kpi_count  max_run       earliest_anomaly_ts  total_above_count  total_below_count  affected_total_points  max_value_in_window  min_value_in_window
0                  adservice                  305       30 2022-03-21 10:00:00+08:00               8883               9038                   9150         1.647523e+12                 -1.0
4                 adservice2                  280       30 2022-03-21 10:00:00+08:00               7876               8004                   8400         1.647822e+12                 -1.0
30               frontend2-0                  107       30 2022-03-21 10:00:00+08:00               3083               3145                   3210         1.647523e+09                  0.0
27                frontend-0                   98       30 2022-03-21 10:00:00+08:00               2734               2698                   2940         1.647523e+09                  0.0
28                frontend-1                   96       30 2022-03-21 10:00:00+08:00               2777               2773                   2880         1.647523e+09                  0.0
29                frontend-2                   82       30 2022-03-21 10:00:00+08:00               2405               2401                   2460         1.647523e+09                  0.0
5               adservice2-0                   77       30 2022-03-21 10:00:00+08:00               2146               2201                   2310         1.647822e+09                  0.0
44   productcatalogservice-1                   73       30 2022-03-21 10:00:00+08:00               2111               2137                   2190         1.647523e+09                  0.0
43   productcatalogservice-0                   72       30 2022-03-21 10:00:00+08:00               2105               2140                   2160         1.647523e+09                  0.0
45   productcatalogservice-2                   72       30 2022-03-21 10:00:00+08:00               2048               2140                   2160         1.647523e+09                  0.0
8              cartservice-1                   71       30 2022-03-21 10:00:00+08:00               2103               2102                   2130         1.647778e+09                  0.0
12         checkoutservice-0                   69       30 2022-03-21 10:00:00+08:00               2015               2036                   2070         1.647523e+09                  0.0
3                adservice-2                   67       30 2022-03-21 10:00:00+08:00               1787               1877                   2010         1.647523e+09                  0.0
7              cartservice-0                   67       30 2022-03-21 10:00:00+08:00               1987               1982                   2010         1.647778e+09                  0.0
10            cartservice2-0                   66       30 2022-03-21 10:00:00+08:00               1953               1959                   1980         1.647810e+09                  0.0
2                adservice-1                   65       30 2022-03-21 10:00:00+08:00               1782               1933                   1950         1.647523e+09                  0.0
1                adservice-0                   64       30 2022-03-21 10:00:00+08:00               1762               1864                   1920         1.647523e+09                  0.0
14         checkoutservice-2                   61       30 2022-03-21 10:00:00+08:00               1803               1807                   1830         1.647523e+09                  0.0
49   recommendationservice-1                   61       30 2022-03-21 10:00:00+08:00               1798               1799                   1830         1.647523e+09                  0.0
51  recommendationservice2-0                   61       30 2022-03-21 10:00:00+08:00               1796               1764                   1830         1.647523e+09                  0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: trace_df, thresholds_df
tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Filter trace rows whose trace_name indicates errors or high latency
trace_focus = trace_df[trace_df['trace_name'].str.contains('error_rate|duration_p95', na=False)].copy()

# Filter to incident window (inclusive)
trace_window = trace_focus[(trace_focus['_ts_dt'] >= window_start) & (trace_focus['_ts_dt'] <= window_end)].copy()

# Prepare thresholds subset: rename kpi_name -> trace_name for join
# thresholds_df should contain columns ['cmdb_id','kpi_name','p95','p5'] from earlier computations
if 'kpi_name' in thresholds_df.columns:
    thresholds_for_traces = thresholds_df.reset_index()[['cmdb_id','kpi_name','p95','p5']].rename(columns={'kpi_name':'trace_name'})
else:
    # Fallback: try to find the name-like column
    possible_name_col = [c for c in thresholds_df.columns if 'name' in c.lower()]
    name_col = possible_name_col[0] if possible_name_col else None
    if name_col:
        thresholds_for_traces = thresholds_df.reset_index()[['cmdb_id', name_col, 'p95','p5']].rename(columns={name_col:'trace_name'})
    else:
        thresholds_for_traces = pd.DataFrame(columns=['cmdb_id','trace_name','p95','p5'])

# Merge thresholds into trace window data (inner join to ensure p95/p5 exist)
trace_with_thresh = trace_window.merge(thresholds_for_traces, on=['cmdb_id','trace_name'], how='inner')

# Helper to compute longest consecutive run of True in boolean array
def longest_run_bool(arr):
    if len(arr) == 0:
        return 0
    max_run = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

records = []
if not trace_with_thresh.empty:
    grouped = trace_with_thresh.groupby(['cmdb_id','trace_name','p95','p5'])
    for (cid, tname, p95, p5), grp in grouped:
        grp_sorted = grp.sort_values('_ts_dt')
        vals = grp_sorted['value'].to_numpy()
        times = grp_sorted['_ts_dt'].to_numpy()
        total_count = int(len(vals))
        above_mask = np.isfinite(p95) & (vals >= p95)
        below_mask = np.isfinite(p5) & (vals <= p5)
        above_count = int(np.count_nonzero(above_mask))
        below_count = int(np.count_nonzero(below_mask))
        earliest_above_ts = pd.to_datetime(times[above_mask].min()) if above_count > 0 else pd.NaT
        earliest_below_ts = pd.to_datetime(times[below_mask].min()) if below_count > 0 else pd.NaT
        longest_run_above = longest_run_bool(above_mask)
        longest_run_below = longest_run_bool(below_mask)
        max_run = max(longest_run_above, longest_run_below)
        records.append({
            'cmdb_id': cid,
            'trace_name': tname,
            'p95': p95,
            'p5': p5,
            'total_count': total_count,
            'above_count': above_count,
            'below_count': below_count,
            'earliest_above_ts': earliest_above_ts,
            'earliest_below_ts': earliest_below_ts,
            'longest_run_above': longest_run_above,
            'longest_run_below': longest_run_below,
            'max_run': max_run
        })

# Build DataFrame and ensure columns exist even if empty
cols = ['cmdb_id','trace_name','p95','p5','total_count','above_count','below_count',
        'earliest_above_ts','earliest_below_ts','longest_run_above','longest_run_below','max_run']
trace_anomalies_df = pd.DataFrame.from_records(records, columns=cols)

# If empty, create empty DataFrame with the expected columns
if trace_anomalies_df.empty:
    trace_anomalies_df = pd.DataFrame(columns=cols)

# Prepare compact preview: top 20 rows sorted by max_run desc then above_count desc
trace_top20 = trace_anomalies_df.sort_values(['max_run','above_count'], ascending=[False, False]).head(20)[
    ['cmdb_id','trace_name','p95','p5','total_count','above_count','below_count',
     'earliest_above_ts','earliest_below_ts','longest_run_above','longest_run_below','max_run']
]

trace_anomalies_df, trace_top20

```
Out[6]:
```
```python
# Plain-English summary (displayed)
summary = (
    "Result summary: No trace anomalies found for trace metrics containing 'error_rate' or 'duration_p95' "
    "within the incident window (2022-03-21 10:00–10:30 Asia/Shanghai). "
    "trace_anomalies_df in the kernel is empty, meaning there were no matched trace series with thresholds "
    "and in-window data that produced breaches by the criteria used."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, p95, p5, total_count, above_count, below_count, earliest_above_ts, earliest_below_ts, longest_run_above, longest_run_below, max_run]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, p95, p5, total_count, above_count, below_count, earliest_above_ts, earliest_below_ts, longest_run_above, longest_run_below, max_run]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse error_logs_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Filter error logs to the incident window (inclusive)
error_window = error_logs_df[(error_logs_df['_ts_dt'] >= window_start) & (error_logs_df['_ts_dt'] <= window_end)].copy()

# 1) Aggregate counts per-minute per-cmdb_id (minute-granularity)
error_window['minute_timestamp_Asia/Shanghai'] = error_window['_ts_dt'].dt.floor('T')
error_minute_counts_df = (
    error_window
    .groupby(['minute_timestamp_Asia/Shanghai', 'cmdb_id'])
    .size()
    .reset_index(name='count')
)

# Top 20 rows sorted by count desc
error_minute_counts_top20 = error_minute_counts_df.sort_values('count', ascending=False).head(20)

# 2) Per-cmdb_id summary: total count, earliest and latest timestamps in window
error_component_summary_df = (
    error_window
    .groupby('cmdb_id')
    .agg(
        total_error_count_in_window = ('message', 'count'),
        earliest_error_ts_in_window = ('_ts_dt', 'min'),
        latest_error_ts_in_window = ('_ts_dt', 'max')
    )
    .reset_index()
)

# Top 20 components by total_error_count_in_window desc
error_component_summary_top20 = error_component_summary_df.sort_values('total_error_count_in_window', ascending=False).head(20)

# 3) For top 3 cmdb_id by total_error_count_in_window, get up to 5 distinct sample messages each (timestamp and message), ordered by timestamp asc
top3_cmdb = error_component_summary_top20['cmdb_id'].head(3).tolist()

sample_rows = []
for cid in top3_cmdb:
    dfc = error_window[error_window['cmdb_id'] == cid].sort_values('_ts_dt')
    # keep distinct messages (first occurrence), then take up to 5 samples
    dfc_unique = dfc.drop_duplicates(subset=['message'], keep='first').head(5)
    if not dfc_unique.empty:
        tmp = dfc_unique.loc[:, ['_ts_dt', 'cmdb_id', 'message']].copy()
        tmp.rename(columns={'_ts_dt': 'timestamp_Asia/Shanghai'}, inplace=True)
        sample_rows.append(tmp)

# Concatenate samples (may be empty)
if sample_rows:
    top3_samples_df = pd.concat(sample_rows, ignore_index=True).sort_values(['cmdb_id','timestamp_Asia/Shanghai'])
else:
    top3_samples_df = pd.DataFrame(columns=['timestamp_Asia/Shanghai','cmdb_id','message'])

# Return compact previews (top-20) while full aggregated DataFrames are stored in kernel variables:
# error_minute_counts_df, error_component_summary_df
error_minute_counts_top20, error_component_summary_top20, top3_samples_df

```
Out[7]:
```
```python
# Build a compact plain-English summary from existing kernel DataFrames:
# error_minute_counts_top20, error_component_summary_top20, top3_samples_df, error_component_summary_df

# Note: these variables were created in previous steps and are present in the kernel.

# Compose summary dynamically to avoid hardcoding
top_minutes = []
for _, r in error_minute_counts_top20.head(5).iterrows():
    ts = r['minute_timestamp_Asia/Shanghai']
    top_minutes.append(f"{ts} | {r['cmdb_id']} | {int(r['count'])}")

top_components_lines = []
for _, r in error_component_summary_top20.iterrows():
    cid = r['cmdb_id']
    total = int(r['total_error_count_in_window'])
    earliest = r['earliest_error_ts_in_window']
    latest = r['latest_error_ts_in_window']
    top_components_lines.append(f"{cid}: total={total}, earliest={earliest}, latest={latest}")

# Build sample messages for top 3 components
samples_lines = []
if not top3_samples_df.empty:
    for cid in top3_samples_df['cmdb_id'].unique():
        samples = top3_samples_df[top3_samples_df['cmdb_id'] == cid].head(5)
        msg_lines = []
        for _, s in samples.iterrows():
            msg_lines.append(f"{s['timestamp_Asia/Shanghai']} | {s['message']}")
        samples_lines.append(f"{cid} samples:\n  " + "\n  ".join(msg_lines))
else:
    samples_lines.append("No sample messages available.")

total_errors = int(error_component_summary_df['total_error_count_in_window'].sum()) if 'error_component_summary_df' in globals() and not error_component_summary_df.empty else None
total_errors_line = f"Total error log entries in window: {total_errors}" if total_errors is not None else ""

summary = (
    "Error logs summary (2022-03-21 10:00–10:30 Asia/Shanghai):\n\n"
    f"{total_errors_line}\n\n"
    "1) Top per-minute error counts (sample top 5):\n"
    + "\n".join(f"- {line}" for line in top_minutes) + "\n\n"
    "2) Per-component totals (top 20 by count):\n"
    + "\n".join(f"- {line}" for line in top_components_lines) + "\n\n"
    "3) Sample messages for top 3 components (up to 5 distinct messages each):\n"
    + ("\n\n".join(samples_lines)) + "\n\n"
    "Findings:\n"
    "- The frontend components dominate error log activity during the window. Top counts:\n"
    "  * frontend-0: 465 errors (first at 10:00:00, last at 10:29:59)\n"
    "  * frontend-2: 436 errors (first at 10:00:02, last at 10:29:54)\n"
    "  * frontend-1: 423 errors (first at 10:00:06, last at 10:29:55)\n"
    "- Peak per-minute counts reach 21 errors in single minutes for frontend instances (e.g., 10:27–10:29).\n"
    "- Sample messages are warnings like 'severity: warning, message: failed to retrieve...'.\n\n"
    "Stored DataFrames in kernel: error_minute_counts_df (per-minute counts), error_component_summary_df (per-component summary), and top3_samples_df (sample messages)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   minute_timestamp_Asia/Shanghai     cmdb_id  count
86      2022-03-21 10:28:00+08:00  frontend-2     21
82      2022-03-21 10:27:00+08:00  frontend-1     21
87      2022-03-21 10:29:00+08:00  frontend-0     21
58      2022-03-21 10:19:00+08:00  frontend-1     20
84      2022-03-21 10:28:00+08:00  frontend-0     20
32      2022-03-21 10:10:00+08:00  frontend-2     19
44      2022-03-21 10:14:00+08:00  frontend-2     19
42      2022-03-21 10:14:00+08:00  frontend-0     18
88      2022-03-21 10:29:00+08:00  frontend-1     18
52      2022-03-21 10:17:00+08:00  frontend-1     18
56      2022-03-21 10:18:00+08:00  frontend-2     18
76      2022-03-21 10:25:00+08:00  frontend-1     18
27      2022-03-21 10:09:00+08:00  frontend-0     17
54      2022-03-21 10:18:00+08:00  frontend-0     17
48      2022-03-21 10:16:00+08:00  frontend-0     17
39      2022-03-21 10:13:00+08:00  frontend-0     17
18      2022-03-21 10:06:00+08:00  frontend-0     17
64      2022-03-21 10:21:00+08:00  frontend-1     17
81      2022-03-21 10:27:00+08:00  frontend-0     17
80      2022-03-21 10:26:00+08:00  frontend-2     17,       cmdb_id  total_error_count_in_window earliest_error_ts_in_window latest_error_ts_in_window
0  frontend-0                          465   2022-03-21 10:00:00+08:00 2022-03-21 10:29:59+08:00
2  frontend-2                          436   2022-03-21 10:00:02+08:00 2022-03-21 10:29:54+08:00
1  frontend-1                          423   2022-03-21 10:00:06+08:00 2022-03-21 10:29:55+08:00,     timestamp_Asia/Shanghai     cmdb_id                                            message
0 2022-03-21 10:00:00+08:00  frontend-0  severity: warning, message: failed to retrieve...
2 2022-03-21 10:00:06+08:00  frontend-1  severity: warning, message: failed to retrieve...
1 2022-03-21 10:00:02+08:00  frontend-2  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing kernel variable: log_df
tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# 1) Filter log_df to window and log_name == 'log.error_count'
log_error_window = log_df[
    (log_df['_ts_dt'] >= window_start) &
    (log_df['_ts_dt'] <= window_end) &
    (log_df['log_name'] == 'log.error_count')
].copy()

# Floor to minute (should already be minute-aligned, but ensure)
log_error_window['minute_timestamp_Asia/Shanghai'] = log_error_window['_ts_dt'].dt.floor('T')

# Aggregate per-minute per-cmdb_id by summing 'value' to produce per-minute error counts
log_error_minute_counts_df = (
    log_error_window
    .groupby(['minute_timestamp_Asia/Shanghai', 'cmdb_id'], as_index=False)['value']
    .sum()
    .rename(columns={'value': 'count'})
)

# If counts are effectively integers, cast to int for compactness
# Use rounding to avoid floating tiny errors
log_error_minute_counts_df['count'] = log_error_minute_counts_df['count'].round().astype(int)

# Top 20 rows sorted by count descending
log_error_minute_counts_top20 = log_error_minute_counts_df.sort_values('count', ascending=False).head(20)[
    ['minute_timestamp_Asia/Shanghai','cmdb_id','count']
]

# 2) Per-cmdb_id summary for these log.error_count rows
log_error_component_summary_df = (
    log_error_minute_counts_df
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_error_count_in_window = ('count', 'sum'),
        earliest_minute_nonzero = ('minute_timestamp_Asia/Shanghai', lambda s: s[log_error_minute_counts_df.loc[s.index,'count'] > 0].min()),
        latest_minute_nonzero = ('minute_timestamp_Asia/Shanghai', lambda s: s[log_error_minute_counts_df.loc[s.index,'count'] > 0].max())
    )
)

# If earliest/latest came out as NaT due to no >0, they remain NaT. Sort top 20 by total_error_count_in_window desc
log_error_component_summary_top20 = log_error_component_summary_df.sort_values('total_error_count_in_window', ascending=False).head(20)[
    ['cmdb_id','total_error_count_in_window','earliest_minute_nonzero','latest_minute_nonzero']
]

# 3) For the four frontend-related cmdb_id values, return per-minute counts for the entire window
frontends = ['frontend-0','frontend-1','frontend-2','frontend']

frontend0_minutes = log_error_minute_counts_df[log_error_minute_counts_df['cmdb_id']=='frontend-0'].sort_values('minute_timestamp_Asia/Shanghai').head(30)
frontend1_minutes = log_error_minute_counts_df[log_error_minute_counts_df['cmdb_id']=='frontend-1'].sort_values('minute_timestamp_Asia/Shanghai').head(30)
frontend2_minutes = log_error_minute_counts_df[log_error_minute_counts_df['cmdb_id']=='frontend-2'].sort_values('minute_timestamp_Asia/Shanghai').head(30)
frontend_minutes  = log_error_minute_counts_df[log_error_minute_counts_df['cmdb_id']=='frontend'].sort_values('minute_timestamp_Asia/Shanghai').head(30)

# Return the stored full DataFrames (kept in kernel) and compact previews as requested
log_error_minute_counts_df, log_error_minute_counts_top20, log_error_component_summary_df, log_error_component_summary_top20, frontend0_minutes, frontend1_minutes, frontend2_minutes, frontend_minutes

```
Out[8]:
```
```python
# Plain-English summary of log.error_count analysis (2022-03-21 10:00–10:30 Asia/Shanghai)
summary = (
    "Log.error_count summary for 2022-03-21 10:00–10:30 (Asia/Shanghai):\n\n"
    "1) Per-minute peaks (top examples):\n"
    "- Highest per-minute counts were observed on frontend instances: frontend-2 (21 errors at 2022-03-21 10:28), "
    "frontend-0 (21 errors at 10:29), and frontend-1 (21 errors at 10:27). Several other minutes reached 17–20 errors on frontends.\n\n"
    "2) Per-component totals (top components):\n"
    "- frontend-0: total = 465 errors (earliest non-zero minute 2022-03-21 10:00, latest 10:29)\n"
    "- frontend-2: total = 436 errors (earliest 10:00, latest 10:29)\n"
    "- frontend-1: total = 423 errors (earliest 10:00, latest 10:29)\n"
    "- Most other components reported zero summed log.error_count during the window.\n\n"
    "3) Frontend per-minute patterns:\n"
    "- frontend-0, frontend-1, and frontend-2 show sustained per-minute error counts throughout the window (sampled 30 rows each). "
    "Notable spike minutes coincide with the per-minute peaks listed above.\n\n"
    "Data retention in kernel:\n"
    "- Per-minute aggregation stored as log_error_minute_counts_df\n"
    "- Per-component summary stored as log_error_component_summary_df\n"
    "- Per-minute tables for frontend-0, frontend-1, frontend-2, and frontend (up to 30 rows each) are available as frontend0_minutes, frontend1_minutes, frontend2_minutes, frontend_minutes\n\n"
    "Interpretation / next steps:\n"
    "- The frontend tier is the clear source of elevated log.error_count during the incident window and should be investigated first (check application logs, traces, and upstream dependencies at those spike minutes).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    minute_timestamp_Asia/Shanghai                  cmdb_id  count
0        2022-03-21 10:00:00+08:00              adservice-0      0
1        2022-03-21 10:00:00+08:00              adservice-1      0
2        2022-03-21 10:00:00+08:00              adservice-2      0
3        2022-03-21 10:00:00+08:00            cartservice-0      0
4        2022-03-21 10:00:00+08:00            cartservice-1      0
..                             ...                      ...    ...
902      2022-03-21 10:29:00+08:00  recommendationservice-1      0
903      2022-03-21 10:29:00+08:00  recommendationservice-2      0
904      2022-03-21 10:29:00+08:00        shippingservice-0      0
905      2022-03-21 10:29:00+08:00        shippingservice-1      0
906      2022-03-21 10:29:00+08:00        shippingservice-2      0

[907 rows x 3 columns],     minute_timestamp_Asia/Shanghai     cmdb_id  count
864      2022-03-21 10:28:00+08:00  frontend-2     21
892      2022-03-21 10:29:00+08:00  frontend-0     21
833      2022-03-21 10:27:00+08:00  frontend-1     21
862      2022-03-21 10:28:00+08:00  frontend-0     20
590      2022-03-21 10:19:00+08:00  frontend-1     20
440      2022-03-21 10:14:00+08:00  frontend-2     19
319      2022-03-21 10:10:00+08:00  frontend-2     19
772      2022-03-21 10:25:00+08:00  frontend-1     18
893      2022-03-21 10:29:00+08:00  frontend-1     18
561      2022-03-21 10:18:00+08:00  frontend-2     18
530      2022-03-21 10:17:00+08:00  frontend-1     18
438      2022-03-21 10:14:00+08:00  frontend-0     18
499      2022-03-21 10:16:00+08:00  frontend-0     17
652      2022-03-21 10:21:00+08:00  frontend-1     17
408      2022-03-21 10:13:00+08:00  frontend-0     17
832      2022-03-21 10:27:00+08:00  frontend-0     17
683      2022-03-21 10:22:00+08:00  frontend-2     17
287      2022-03-21 10:09:00+08:00  frontend-0     17
197      2022-03-21 10:06:00+08:00  frontend-0     17
469      2022-03-21 10:15:00+08:00  frontend-1     17,                     cmdb_id  total_error_count_in_window earliest_minute_nonzero latest_minute_nonzero
0               adservice-0                            0                     NaT                   NaT
1               adservice-1                            0                     NaT                   NaT
2               adservice-2                            0                     NaT                   NaT
3             cartservice-0                            0                     NaT                   NaT
4             cartservice-1                            0                     NaT                   NaT
..                      ...                          ...                     ...                   ...
26  recommendationservice-2                            0                     NaT                   NaT
27             redis-cart-0                            0                     NaT                   NaT
28        shippingservice-0                            0                     NaT                   NaT
29        shippingservice-1                            0                     NaT                   NaT
30        shippingservice-2                            0                     NaT                   NaT

[31 rows x 4 columns],               cmdb_id  total_error_count_in_window   earliest_minute_nonzero     latest_minute_nonzero
15         frontend-0                          465 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
17         frontend-2                          436 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
16         frontend-1                          423 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
3       cartservice-0                            0                       NaT                       NaT
0         adservice-0                            0                       NaT                       NaT
1         adservice-1                            0                       NaT                       NaT
2         adservice-2                            0                       NaT                       NaT
7   checkoutservice-1                            0                       NaT                       NaT
8   checkoutservice-2                            0                       NaT                       NaT
9   currencyservice-0                            0                       NaT                       NaT
10  currencyservice-1                            0                       NaT                       NaT
11  currencyservice-2                            0                       NaT                       NaT
4       cartservice-1                            0                       NaT                       NaT
5       cartservice-2                            0                       NaT                       NaT
6   checkoutservice-0                            0                       NaT                       NaT
14     emailservice-2                            0                       NaT                       NaT
13     emailservice-1                            0                       NaT                       NaT
12     emailservice-0                            0                       NaT                       NaT
18   paymentservice-0                            0                       NaT                       NaT
19   paymentservice-1                            0                       NaT                       NaT,     minute_timestamp_Asia/Shanghai     cmdb_id  count
15       2022-03-21 10:00:00+08:00  frontend-0     11
46       2022-03-21 10:01:00+08:00  frontend-0     13
76       2022-03-21 10:02:00+08:00  frontend-0     13
106      2022-03-21 10:03:00+08:00  frontend-0     11
136      2022-03-21 10:04:00+08:00  frontend-0     16
..                             ...         ...    ...
771      2022-03-21 10:25:00+08:00  frontend-0     16
802      2022-03-21 10:26:00+08:00  frontend-0     14
832      2022-03-21 10:27:00+08:00  frontend-0     17
862      2022-03-21 10:28:00+08:00  frontend-0     20
892      2022-03-21 10:29:00+08:00  frontend-0     21

[30 rows x 3 columns],     minute_timestamp_Asia/Shanghai     cmdb_id  count
16       2022-03-21 10:00:00+08:00  frontend-1     14
47       2022-03-21 10:01:00+08:00  frontend-1     12
77       2022-03-21 10:02:00+08:00  frontend-1     13
107      2022-03-21 10:03:00+08:00  frontend-1     14
137      2022-03-21 10:04:00+08:00  frontend-1     11
..                             ...         ...    ...
772      2022-03-21 10:25:00+08:00  frontend-1     18
803      2022-03-21 10:26:00+08:00  frontend-1     10
833      2022-03-21 10:27:00+08:00  frontend-1     21
863      2022-03-21 10:28:00+08:00  frontend-1     10
893      2022-03-21 10:29:00+08:00  frontend-1     18

[30 rows x 3 columns],     minute_timestamp_Asia/Shanghai     cmdb_id  count
17       2022-03-21 10:00:00+08:00  frontend-2     11
48       2022-03-21 10:01:00+08:00  frontend-2     16
78       2022-03-21 10:02:00+08:00  frontend-2     13
108      2022-03-21 10:03:00+08:00  frontend-2     14
138      2022-03-21 10:04:00+08:00  frontend-2     12
..                             ...         ...    ...
773      2022-03-21 10:25:00+08:00  frontend-2     14
804      2022-03-21 10:26:00+08:00  frontend-2     17
834      2022-03-21 10:27:00+08:00  frontend-2     15
864      2022-03-21 10:28:00+08:00  frontend-2     21
894      2022-03-21 10:29:00+08:00  frontend-2      9

[30 rows x 3 columns], Empty DataFrame
Columns: [minute_timestamp_Asia/Shanghai, cmdb_id, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: log_error_minute_counts_df, frontend0_minutes, frontend1_minutes, frontend2_minutes
tz = pytz.timezone('Asia/Shanghai')

# Define incident window bounds (inclusive)
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Ensure we have the per-minute counts DataFrame; fallback to combining per-frontend minute frames if needed
if 'log_error_minute_counts_df' in globals() and not log_error_minute_counts_df.empty:
    per_min_df = log_error_minute_counts_df.copy()
else:
    per_min_df = pd.concat([frontend0_minutes, frontend1_minutes, frontend2_minutes], ignore_index=True)

# Focus on the three frontends
frontends = ['frontend-0','frontend-1','frontend-2']

# Build full minute index for the window (floor to minutes). Include up to 10:29 as typical per-minute timestamps.
minute_index = pd.date_range(window_start, window_end, freq='T', tz=tz)

records = []
for fid in frontends:
    # Extract counts for this frontend
    df_f = per_min_df[per_min_df['cmdb_id'] == fid].copy()
    # Ensure minute column is tz-aware datetime
    if 'minute_timestamp_Asia/Shanghai' in df_f.columns:
        df_f = df_f.rename(columns={'minute_timestamp_Asia/Shanghai':'minute_ts'})
    else:
        df_f = df_f.rename(columns={'minute_timestamp_Asia/Shanghai':'minute_ts'}) if 'minute_timestamp_Asia/Shanghai' in df_f.columns else df_f
    # Some sources used '_ts_dt' before; ensure we use minute timestamps
    if 'minute_ts' not in df_f.columns and '_ts_dt' in df_f.columns:
        df_f['minute_ts'] = df_f['_ts_dt'].dt.floor('T')
    # If minute column still missing, try index
    if 'minute_ts' not in df_f.columns:
        # nothing to do, skip
        continue
    df_f = df_f.set_index('minute_ts').sort_index()
    # Reindex to full minute_index, filling missing minutes with 0
    df_f = df_f.reindex(minute_index, fill_value=0)
    df_f = df_f.rename_axis('minute_ts').reset_index()
    # Ensure count column exists (may be named 'count' or 'value')
    if 'count' in df_f.columns:
        df_f['count'] = df_f['count'].astype(int)
    elif 'value' in df_f.columns:
        df_f['count'] = df_f['value'].round().astype(int)
    else:
        df_f['count'] = 0

    # Compute previous minute counts by shifting
    df_f['previous_minute_timestamp'] = df_f['minute_ts'] - pd.Timedelta(minutes=1)
    df_f['previous_count'] = df_f['count'].shift(1).fillna(0).astype(int)
    df_f['current_count'] = df_f['count'].astype(int)
    df_f['delta'] = df_f['current_count'] - df_f['previous_count']

    # percent_change: if previous_count == 0 and current_count > 0 -> inf; if both zero -> NaN; else compute percent
    def pct(row):
        prev = row['previous_count']
        cur = row['current_count']
        if prev == 0:
            return np.inf if cur > 0 else np.nan
        else:
            return (cur - prev) / prev * 100.0

    df_f['percent_change'] = df_f.apply(pct, axis=1)

    # Add frontend id
    df_f['frontend_id'] = fid

    # Keep relevant columns
    df_out = df_f[['frontend_id','minute_ts','previous_minute_timestamp','previous_count','current_count','delta','percent_change']].copy()
    records.append(df_out)

# Concatenate per-frontend diffs
if records:
    frontend_error_diffs_df = pd.concat(records, ignore_index=True)
else:
    frontend_error_diffs_df = pd.DataFrame(columns=['frontend_id','minute_ts','previous_minute_timestamp','previous_count','current_count','delta','percent_change'])

# Store full per-minute diff table in kernel variable frontend_error_diffs_df
# Now pick top 3 positive increases per frontend (largest delta), then combine and sort by delta desc, limit 9 rows
top_per_frontend = frontend_error_diffs_df[frontend_error_diffs_df['delta'] > 0].copy()
top_per_frontend = top_per_frontend.sort_values(['frontend_id','delta'], ascending=[True, False]).groupby('frontend_id').head(3)
frontend_top9 = top_per_frontend.sort_values('delta', ascending=False).head(9)

# Rename minute columns to requested names for output clarity
frontend_top9 = frontend_top9.rename(columns={
    'minute_ts': 'minute_timestamp_Asia/Shanghai',
    'previous_minute_timestamp': 'previous_minute_timestamp',
    'previous_count': 'previous_count',
    'current_count': 'current_count'
})[
    ['frontend_id','minute_timestamp_Asia/Shanghai','previous_minute_timestamp','previous_count','current_count','delta','percent_change']
]

# Display compact preview (top 9 rows). Full table retained in frontend_error_diffs_df.
frontend_top9

```
Out[9]:
```
```python
# Build a plain-English summary from the in-kernel per-minute diff table frontend_error_diffs_df
import pandas as pd
import numpy as np

# Use existing frontend_error_diffs_df from kernel
df = frontend_error_diffs_df.copy()

summary_lines = []
if df.empty:
    summary_lines.append("No per-minute frontend error diffs available in frontend_error_diffs_df.")
else:
    frontends = ['frontend-0','frontend-1','frontend-2']
    summary_lines.append("Top 3 minute-by-minute positive increases per frontend (2022-03-21 10:00–10:30 Asia/Shanghai):")
    for fid in frontends:
        sub = df[df['frontend_id']==fid].sort_values('delta', ascending=False)
        sub_pos = sub[sub['delta']>0].head(3)
        if sub_pos.empty:
            summary_lines.append(f"- {fid}: no positive increases found.")
            continue
        summary_lines.append(f"- {fid}:")
        for _, r in sub_pos.iterrows():
            minute = pd.Timestamp(r['minute_ts'])
            prev_min = pd.Timestamp(r['previous_minute_timestamp'])
            prev = int(r['previous_count'])
            cur = int(r['current_count'])
            delta = int(r['delta'])
            pct = r['percent_change']
            pct_str = "inf" if np.isinf(pct) else (f"{pct:.2f}%" if pd.notnull(pct) else "N/A")
            summary_lines.append(
                f"    {minute} (vs {prev_min}): previous={prev}, current={cur}, delta=+{delta}, percent_change={pct_str}"
            )
    # Also highlight the single largest increases overall
    top_overall = df[df['delta']>0].sort_values('delta', ascending=False).head(3)
    if not top_overall.empty:
        summary_lines.append("\nLargest overall positive increases across frontends:")
        for _, r in top_overall.iterrows():
            summary_lines.append(
                f"- {r['frontend_id']} at {pd.Timestamp(r['minute_ts'])}: delta=+{int(r['delta'])}, previous={int(r['previous_count'])}, current={int(r['current_count'])}"
            )

    summary_lines.append("\nNotes:")
    summary_lines.append("- Several frontends show an initial spike at 10:00 where previous minute is zero (percent_change = inf).")
    summary_lines.append("- Notable non-initial spikes: frontend-1 at 10:27 (+11, 110%), frontend-1 at 10:19 (+8, ~66.7%), frontend-2 at 10:14 (+7, ~58.3%).")
    summary_lines.append("- Full per-minute diff table is stored in kernel as frontend_error_diffs_df for further inspection.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

frontend_id minute_timestamp_Asia/Shanghai previous_minute_timestamp  previous_count  current_count  delta  percent_change
31  frontend-1      2022-03-21 10:00:00+08:00 2022-03-21 09:59:00+08:00               0             14     14             inf
0   frontend-0      2022-03-21 10:00:00+08:00 2022-03-21 09:59:00+08:00               0             11     11             inf
58  frontend-1      2022-03-21 10:27:00+08:00 2022-03-21 10:26:00+08:00              10             21     11      110.000000
62  frontend-2      2022-03-21 10:00:00+08:00 2022-03-21 09:59:00+08:00               0             11     11             inf
50  frontend-1      2022-03-21 10:19:00+08:00 2022-03-21 10:18:00+08:00              12             20      8       66.666667
76  frontend-2      2022-03-21 10:14:00+08:00 2022-03-21 10:13:00+08:00              12             19      7       58.333333
72  frontend-2      2022-03-21 10:10:00+08:00 2022-03-21 10:09:00+08:00              13             19      6       46.153846
4   frontend-0      2022-03-21 10:04:00+08:00 2022-03-21 10:03:00+08:00              11             16      5       45.454545
18  frontend-0      2022-03-21 10:18:00+08:00 2022-03-21 10:17:00+08:00              14             17      3       21.428571```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import re

# Reuse thresholds_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Target frontends and keyword substrings (case-insensitive)
frontends = ["frontend-0","frontend-1","frontend-2"]
keywords = [
    'cpu','memory','mem','process','thread','io','read','write','disk','fs',
    'network','packet','latency','latency_p95','error','oom','terminate','uptime'
]

# Build a regex pattern that matches any of the keywords (case-insensitive)
# Ensure we escape keywords that may contain regex-special chars
pattern = re.compile("|".join(re.escape(k) for k in keywords), flags=re.IGNORECASE)

# Defensive: ensure expected columns exist in thresholds_df
expected_cols = ['cmdb_id','kpi_name','p95','p5','total_count','min_timestamp','max_timestamp']
available_cols = thresholds_df.columns.tolist()

# Map possible alternate timestamp column names if necessary
ts_min_col = 'min_timestamp' if 'min_timestamp' in thresholds_df.columns else ('min_ts' if 'min_ts' in thresholds_df.columns else None)
ts_max_col = 'max_timestamp' if 'max_timestamp' in thresholds_df.columns else ('max_ts' if 'max_ts' in thresholds_df.columns else None)

# Use fallback names if total_count/p95/p5 differ (they should exist)
count_col = 'total_count' if 'total_count' in thresholds_df.columns else ('total_count' if 'total_count' in thresholds_df.columns else None)
p95_col = 'p95' if 'p95' in thresholds_df.columns else None
p5_col = 'p5' if 'p5' in thresholds_df.columns else None

# Filter thresholds_df for target frontends and kpi_name matching pattern
mask_cmdb = thresholds_df['cmdb_id'].isin(frontends)
mask_kpi = thresholds_df['kpi_name'].astype(str).apply(lambda s: bool(pattern.search(s)))
frontend_resource_kpis_df = thresholds_df[mask_cmdb & mask_kpi].copy()

# Normalize column names for output selection (ensure they exist)
# If some expected numeric cols are missing, create them with NaN to keep schema consistent
if p95_col not in frontend_resource_kpis_df.columns:
    frontend_resource_kpis_df['p95'] = pd.NA
if p5_col not in frontend_resource_kpis_df.columns:
    frontend_resource_kpis_df['p5'] = pd.NA
if count_col not in frontend_resource_kpis_df.columns:
    frontend_resource_kpis_df['total_count'] = pd.NA
if ts_min_col is None:
    frontend_resource_kpis_df['min_timestamp'] = pd.NaT
else:
    # ensure timezone-aware (should already be)
    frontend_resource_kpis_df['min_timestamp'] = pd.to_datetime(frontend_resource_kpis_df[ts_min_col]).dt.tz_convert(tz) if pd.api.types.is_datetime64_any_dtype(frontend_resource_kpis_df[ts_min_col]) or frontend_resource_kpis_df[ts_min_col].dtype == object else frontend_resource_kpis_df[ts_min_col]
if ts_max_col is None:
    frontend_resource_kpis_df['max_timestamp'] = pd.NaT
else:
    frontend_resource_kpis_df['max_timestamp'] = pd.to_datetime(frontend_resource_kpis_df[ts_max_col]).dt.tz_convert(tz) if pd.api.types.is_datetime64_any_dtype(frontend_resource_kpis_df[ts_max_col]) or frontend_resource_kpis_df[ts_max_col].dtype == object else frontend_resource_kpis_df[ts_max_col]

# Select and order output columns
output_cols = ['cmdb_id','kpi_name','total_count','p95','p5','min_timestamp','max_timestamp']
# If total_count originally existed under a different name, align it
if 'total_count' not in frontend_resource_kpis_df.columns and 'total_count' in thresholds_df.columns:
    frontend_resource_kpis_df['total_count'] = thresholds_df['total_count']

# Final compact table (limit to 100 rows)
frontend_resource_kpis_preview = frontend_resource_kpis_df.loc[:, output_cols].head(100)

# Store full result in kernel variable frontend_resource_kpis_df (already assigned)
frontend_resource_kpis_df, frontend_resource_kpis_preview

```
Out[10]:
```
```python
# Plain-English summary string based on frontend_resource_kpis_df in kernel
# Displayed as a single compact string variable per instructions.

df = frontend_resource_kpis_df  # reused from kernel

n_rows = int(df.shape[0])
time_min = df['min_timestamp'].min() if 'min_timestamp' in df.columns else None
time_max = df['max_timestamp'].max() if 'max_timestamp' in df.columns else None

# sample distinct KPI names (up to 10)
sample_kpis = df['kpi_name'].dropna().unique()[:10].tolist()

# top 5 KPIs by p95 value (if numeric)
top5_by_p95 = []
if 'p95' in df.columns:
    top5 = df.sort_values('p95', ascending=False).head(5)[['kpi_name','p95']].dropna()
    for _, r in top5.iterrows():
        top5_by_p95.append(f"{r['kpi_name']} (p95={r['p95']})")

summary = (
    f"Frontend resource KPI extraction summary:\n\n"
    f"- Total matching KPI rows for frontend-0/1/2: {n_rows} rows.\n"
    f"- Time coverage for these KPIs: {time_min} to {time_max} (Asia/Shanghai).\n"
    f"- Typical per-series sample count: many series show total_count = 30 (full 30-minute window).\n"
    f"- Matched KPI types include CPU metrics (e.g., container_cpu_*), memory/oom/process/thread metrics, I/O and disk/fs metrics, network/mesh/request latency metrics, and error-related keys.\n"
    f"- Example KPI names (up to 10): {', '.join(sample_kpis)}\n"
    f"- Top examples by P95 (up to 5): " + (", ".join(top5_by_p95) if top5_by_p95 else "N/A") + "\n\n"
    "Notes / guidance:\n"
    "- These KPIs provide coverage of resource (CPU/memory/disk/IO) and network/mesh latency for the frontends during the incident window; most have full-sample coverage.\n"
    "- Use the stored DataFrame frontend_resource_kpis_df for detailed inspection (per-series p95, p5, counts and exact timestamps) to prioritize which resource or latency KPIs to investigate further."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name          p95         p90         p50          p5  total_count             min_timestamp             max_timestamp  in_window_count
2522  frontend-0         container.node-5.container_cpu_cfs_periods   190.075000   183.35000   162.50000   130.50000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
2523  frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.000000     0.00000     0.00000     0.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
2524  frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.000000     0.00000     0.00000     0.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
2525  frontend-0    container.node-5.container_cpu_load_average_10s     0.000000     0.00000     0.00000     0.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
2526  frontend-0      container.node-5.container_cpu_system_seconds     0.190500     0.17150     0.14250     0.11725           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
...          ...                                                ...          ...         ...         ...         ...          ...                       ...                       ...              ...
3003  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.019714     0.01848     0.01575     0.00945           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
3004  frontend-2  mesh.source.frontend.shippingservice.istio_req...    19.100000    17.55000    14.50000    10.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
3005  frontend-2  mesh.source.frontend.shippingservice.istio_req...    19.100000    17.55000    14.50000    10.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
3006  frontend-2  mesh.source.frontend.shippingservice.istio_res...  2910.125000  2728.00000  2325.00000  1395.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30
3007  frontend-2  mesh.source.frontend.shippingservice.istio_res...    19.100000    17.55000    14.50000    10.00000           30 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00               30

[459 rows x 10 columns],          cmdb_id                                           kpi_name  total_count       p95         p5             min_timestamp             max_timestamp
2522  frontend-0         container.node-5.container_cpu_cfs_periods           30  190.0750  130.50000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2523  frontend-0  container.node-5.container_cpu_cfs_throttled_p...           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2524  frontend-0  container.node-5.container_cpu_cfs_throttled_s...           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2525  frontend-0    container.node-5.container_cpu_load_average_10s           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2526  frontend-0      container.node-5.container_cpu_system_seconds           30    0.1905    0.11725 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
...          ...                                                ...          ...       ...        ...                       ...                       ...
2626  frontend-0  mesh.source.frontend.cartservice.istio_request...           30   75.2750   54.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2627  frontend-0  mesh.source.frontend.cartservice.istio_request...           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2628  frontend-0  mesh.source.frontend.cartservice.istio_request...           30   75.2750   54.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2629  frontend-0  mesh.source.frontend.cartservice.istio_request...           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00
2630  frontend-0  mesh.source.frontend.cartservice.istio_respons...           30    0.0000    0.00000 2022-03-21 10:00:00+08:00 2022-03-21 10:29:00+08:00

[100 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_df, thresholds_df, frontend_resource_kpis_df
tz = pytz.timezone('Asia/Shanghai')

# Incident window (inclusive) in Asia/Shanghai
window_start = pd.Timestamp('2022-03-21 10:00:00').tz_localize(tz)
window_end = pd.Timestamp('2022-03-21 10:30:00').tz_localize(tz)

# Ensure metric_window is available (filter metric_df to the incident window)
metric_window = metric_df[(metric_df['_ts_dt'] >= window_start) & (metric_df['_ts_dt'] <= window_end)].copy()

# Use the frontend_resource_kpis_df as the list of series to analyze
series_list = frontend_resource_kpis_df[['cmdb_id','kpi_name','p95','p5']].drop_duplicates().reset_index(drop=True)

# Helper to compute longest consecutive True run in a boolean array
def longest_run_bool(arr):
    if len(arr) == 0:
        return 0
    max_run = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

records = []
# Iterate over each target series (small enough)
for _, s in series_list.iterrows():
    cid = s['cmdb_id']
    kpi = s['kpi_name']
    p95 = s['p95']
    p5 = s['p5']

    grp = metric_window[(metric_window['cmdb_id']==cid) & (metric_window['kpi_name']==kpi)].sort_values('_ts_dt')

    if grp.shape[0] > 0:
        vals = grp['value'].to_numpy()
        times = grp['_ts_dt'].to_numpy()
        total_count = int(len(vals))
        above_mask = np.isfinite(p95) & (vals >= p95)
        below_mask = np.isfinite(p5) & (vals <= p5)
        above_count = int(np.count_nonzero(above_mask))
        below_count = int(np.count_nonzero(below_mask))
        earliest_above_ts = pd.to_datetime(times[above_mask].min()) if above_count>0 else pd.NaT
        earliest_below_ts = pd.to_datetime(times[below_mask].min()) if below_count>0 else pd.NaT
        longest_run_above = longest_run_bool(above_mask)
        longest_run_below = longest_run_bool(below_mask)
        max_value_in_window = float(np.nanmax(vals)) if np.isfinite(vals).any() else np.nan
        min_value_in_window = float(np.nanmin(vals)) if np.isfinite(vals).any() else np.nan
    else:
        total_count = 0
        above_count = 0
        below_count = 0
        earliest_above_ts = pd.NaT
        earliest_below_ts = pd.NaT
        longest_run_above = 0
        longest_run_below = 0
        max_value_in_window = np.nan
        min_value_in_window = np.nan

    # breach severities
    if above_count > 0 and pd.notnull(max_value_in_window) and pd.notnull(p95):
        breach_severity_above = (max_value_in_window - p95) / (abs(p95) + 1e-9)
    else:
        breach_severity_above = np.nan

    if below_count > 0 and pd.notnull(min_value_in_window) and pd.notnull(p5):
        breach_severity_below = (p5 - min_value_in_window) / (abs(p5) + 1e-9)
    else:
        breach_severity_below = np.nan

    # overall breach_severity: pick the larger magnitude severity if both exist, else the one that exists
    sev_candidates = [v for v in [breach_severity_above, breach_severity_below] if pd.notnull(v)]
    breach_severity = float(max(sev_candidates)) if sev_candidates else np.nan

    records.append({
        'cmdb_id': cid,
        'kpi_name': kpi,
        'p95': p95,
        'p5': p5,
        'total_count': total_count,
        'above_count': above_count,
        'below_count': below_count,
        'earliest_above_ts': earliest_above_ts,
        'earliest_below_ts': earliest_below_ts,
        'longest_run_above': int(longest_run_above),
        'longest_run_below': int(longest_run_below),
        'max_value_in_window': max_value_in_window,
        'min_value_in_window': min_value_in_window,
        'breach_severity': breach_severity
    })

# Build DataFrame and compute max_run
frontend_kpi_anomalies_df = pd.DataFrame.from_records(records)
frontend_kpi_anomalies_df['max_run'] = frontend_kpi_anomalies_df[['longest_run_above','longest_run_below']].max(axis=1)

# Compact preview: top 20 rows sorted by max_run desc
frontend_top20 = frontend_kpi_anomalies_df.sort_values(['max_run'], ascending=False).head(20)[
    ['cmdb_id','kpi_name','p95','p5','total_count','above_count','below_count',
     'earliest_above_ts','earliest_below_ts','longest_run_above','longest_run_below',
     'max_run','max_value_in_window','min_value_in_window','breach_severity']
]

# Expose variables for inspection (kernel retains full frontend_kpi_anomalies_df)
frontend_kpi_anomalies_df, frontend_top20

```
Out[11]:
```
```python
# Compact plain-English summary based on frontend_kpi_anomalies_df in the kernel
import pandas as pd
import numpy as np

# Use existing dataframe from kernel
df = frontend_kpi_anomalies_df if 'frontend_kpi_anomalies_df' in globals() else None

if df is None or df.empty:
    summary = "No frontend KPI anomaly data available in frontend_kpi_anomalies_df."
else:
    total_series = int(df.shape[0])
    per_frontend_counts = df['cmdb_id'].value_counts().to_dict()
    series_with_data = int((df['total_count'] > 0).sum())
    series_with_above = int((df['above_count'] > 0).sum())
    series_with_below = int((df['below_count'] > 0).sum())
    series_long_runs = int((df['max_run'] >= 3).sum())
    max_run_overall = int(df['max_run'].max())

    # Top 5 by max_run (tie-breaker: breach_severity)
    top_by_run = df.sort_values(['max_run','breach_severity'], ascending=[False, False]).head(5)[
        ['cmdb_id','kpi_name','max_run','above_count','below_count','breach_severity']
    ].fillna('N/A').to_dict('records')

    # Top 5 by breach_severity (non-null), descending
    top_by_sev = df[df['breach_severity'].notnull()].sort_values('breach_severity', ascending=False).head(5)[
        ['cmdb_id','kpi_name','max_run','above_count','below_count','breach_severity']
    ].fillna('N/A').to_dict('records')

    lines = []
    lines.append("Frontend resource KPI anomaly summary (2022-03-21 10:00–10:30 Asia/Shanghai):")
    lines.append(f"- Total frontend KPI series analyzed: {total_series}")
    lines.append(f"- Series with any in-window samples: {series_with_data}")
    lines.append(f"- Series with values >= global P95 at least once: {series_with_above}")
    lines.append(f"- Series with values <= global P5 at least once: {series_with_below}")
    lines.append(f"- Series with sustained runs (max_run >= 3 minutes): {series_long_runs}")
    lines.append(f"- Longest consecutive run observed among these series: {max_run_overall} minutes")
    lines.append("")
    lines.append("Per-frontend series counts:")
    for fid, cnt in per_frontend_counts.items():
        lines.append(f"  - {fid}: {int(cnt)} series")
    lines.append("")

    lines.append("Top examples by longest consecutive run (max_run):")
    for r in top_by_run:
        sev = f"{r['breach_severity']:.3g}" if isinstance(r['breach_severity'], (int,float)) and not pd.isna(r['breach_severity']) else r['breach_severity']
        lines.append(f"  - {r['cmdb_id']} | {r['kpi_name']} | max_run={int(r['max_run'])}min | above={int(r['above_count'])} | below={int(r['below_count'])} | severity={sev}")

    lines.append("")
    lines.append("Top examples by breach_severity (magnitude):")
    if top_by_sev:
        for r in top_by_sev:
            sev = f"{r['breach_severity']:.3g}" if isinstance(r['breach_severity'], (int,float)) and not pd.isna(r['breach_severity']) else r['breach_severity']
            lines.append(f"  - {r['cmdb_id']} | {r['kpi_name']} | severity={sev} | max_run={int(r['max_run'])}min | above={int(r['above_count'])} | below={int(r['below_count'])}")
    else:
        lines.append("  - (no non-null breach_severity values)")

    lines.append("")
    lines.append("Key observations and guidance:")
    lines.append("- Many frontend resource and mesh KPIs show full 30-minute coverage and repeated breaches; several series have sustained consecutive breaches (max_run up to 30).")
    lines.append("- Some CPU throttling and mesh/request KPIs show persistent breaches (e.g., container CPU throttling metrics and many mesh request/response metrics).")
    lines.append("- Breach severity values can be very large when P95 or P5 are zero (watch for divisions by near-zero producing large ratios).")
    lines.append("- Next steps: prioritize KPIs with long max_run and high breach_severity for each frontend, investigate corresponding timestamps (earliest_above/earliest_below) and correlate with frontend error logs and traces.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name          p95          p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below  max_value_in_window  min_value_in_window  breach_severity  max_run
0    frontend-0         container.node-5.container_cpu_cfs_periods   190.075000   130.50000           30            2            3 2022-03-21 10:06:00+08:00 2022-03-21 10:00:00+08:00                  1                  2           200.000000           109.500000     1.609195e-01        2
1    frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.000000     0.00000           30           30           29 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 29            11.500000             0.000000     1.150000e+10       30
2    frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.000000     0.00000           30           30           29 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 29             1.740558             0.000000     1.740558e+09       30
3    frontend-0    container.node-5.container_cpu_load_average_10s     0.000000     0.00000           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30             0.000000             0.000000     0.000000e+00       30
4    frontend-0      container.node-5.container_cpu_system_seconds     0.190500     0.11725           30            2            2 2022-03-21 10:06:00+08:00 2022-03-21 10:02:00+08:00                  1                  2             0.270000             0.110000     4.173228e-01        2
..          ...                                                ...          ...         ...          ...          ...          ...                       ...                       ...                ...                ...                  ...                  ...              ...      ...
454  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.019714     0.00945           30            2            2 2022-03-21 10:19:00+08:00 2022-03-21 10:05:00+08:00                  1                  1             0.022050             0.008925     1.185086e-01        1
455  frontend-2  mesh.source.frontend.shippingservice.istio_req...    19.100000    10.00000           30            2            3 2022-03-21 10:19:00+08:00 2022-03-21 10:05:00+08:00                  1                  1            20.500000             9.000000     1.000000e-01        1
456  frontend-2  mesh.source.frontend.shippingservice.istio_req...    19.100000    10.00000           30            2            3 2022-03-21 10:19:00+08:00 2022-03-21 10:05:00+08:00                  1                  1            20.500000             9.000000     1.000000e-01        1
457  frontend-2  mesh.source.frontend.shippingservice.istio_res...  2910.125000  1395.00000           30            2            3 2022-03-21 10:19:00+08:00 2022-03-21 10:05:00+08:00                  1                  1          3255.000000          1317.500000     1.185087e-01        1
458  frontend-2  mesh.source.frontend.shippingservice.istio_res...    19.100000    10.00000           30            2            3 2022-03-21 10:19:00+08:00 2022-03-21 10:05:00+08:00                  1                  1            20.500000             9.000000     1.000000e-01        1

[459 rows x 15 columns],         cmdb_id                                           kpi_name    p95     p5  total_count  above_count  below_count         earliest_above_ts         earliest_below_ts  longest_run_above  longest_run_below  max_run  max_value_in_window  min_value_in_window  breach_severity
450  frontend-2  mesh.source.frontend.recommendationservice.ist...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
409  frontend-2  mesh.source.frontend.cartservice.istio_request...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
411  frontend-2  mesh.source.frontend.cartservice.istio_respons...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
401  frontend-2  mesh.source.frontend.adservice.istio_response_...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
404  frontend-2  mesh.source.frontend.cartservice.istio_request...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
406  frontend-2  mesh.source.frontend.cartservice.istio_request...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
394  frontend-2  mesh.source.frontend.adservice.istio_requests....    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
397  frontend-2  mesh.source.frontend.adservice.istio_requests....    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
398  frontend-2  mesh.source.frontend.adservice.istio_response_...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
385  frontend-2  mesh.source.frontend.adservice.istio_request_b...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
388  frontend-2  mesh.source.frontend.adservice.istio_request_b...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
389  frontend-2  mesh.source.frontend.adservice.istio_request_d...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
392  frontend-2  mesh.source.frontend.adservice.istio_request_d...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
58   frontend-0  mesh.destination.unknown.frontend.istio_reques...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
59   frontend-0  mesh.destination.unknown.frontend.istio_reques...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
62   frontend-0  mesh.destination.unknown.frontend.istio_reques...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
63   frontend-0  mesh.destination.unknown.frontend.istio_reques...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
49   frontend-0    container.node-5.container_spec_memory_limit_MB  256.0  256.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                256.0                256.0              0.0
50   frontend-0  container.node-5.container_spec_memory_reserva...    0.0    0.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                  0.0                  0.0              0.0
51   frontend-0  container.node-5.container_spec_memory_swap_li...  256.0  256.0           30           30           30 2022-03-21 10:00:00+08:00 2022-03-21 10:00:00+08:00                 30                 30       30                256.0                256.0              0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```